In [1]:
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd
from apt_detection.device_telemetry.evaluate import score_report

PROC = Path("../../data/processed/device_telemetry")
MODELS = Path("../../models/device_telemetry")
cfg = json.loads(Path("../../configs/device_telemetry_final.json").read_text())

lgbm = joblib.load(MODELS / "lightgbm.joblib")
et = joblib.load(MODELS / "extra_trees.joblib")
svm = joblib.load(MODELS / "one_class_svm.joblib")

test = pd.read_parquet(PROC / "phase2_heldout_test.parquet")
y_t, kind_t = test["label"].to_numpy(), test["kind"].to_numpy()
print(len(test), test["kind"].value_counts().to_dict())
print("threshold:", round(cfg["threshold"], 4), " features:", len(cfg["features"]))

1559 {'normal': 1516, 'other': 36, 'flood': 7}
threshold: 0.6638  features: 45


In [2]:
X_t = test[cfg["features"]]
score_t = (lgbm.predict_proba(X_t)[:, 1] + et.predict_proba(X_t)[:, 1]) / 2
alert_t = score_t >= cfg["threshold"]

test_hours = (test["t_start"].max() - test["t_start"].min()) / 3600
final = score_report(y_t, score_t, kind_t, threshold=cfg["threshold"])
final["false_alerts_per_day"] = final["fpr"] * (kind_t == "normal").sum() / test_hours * 24
final["other_caught"] = int(alert_t[kind_t == "other"].sum())
final["normal_flagged"] = int(alert_t[kind_t == "normal"].sum())
pd.Series(final).round(4)

recall_all               0.6977
recall_flood             1.0000
recall_other             0.6389
fpr                      0.0000
precision                1.0000
pr_auc_other             0.9414
false_alerts_per_day     0.0000
other_caught            23.0000
normal_flagged           0.0000
dtype: float64

In [4]:
F2 = pd.read_parquet(PROC / "phase2_features.parquet")
cutoff = pd.Timestamp(cfg["cutoff_timestamp_utc"], tz="UTC").timestamp()
seen_tactics = set(F2.loc[(F2["t_start"] < cutoff) & (F2["label"] == 1), "tactic"])

attacks = test[test["label"] == 1].assign(caught=alert_t[y_t == 1], score=score_t[y_t == 1])
by_tactic = attacks.groupby("tactic").agg(processes=("caught", "size"),
                                          caught=("caught", "sum"),
                                          median_score=("score", "median"))
by_tactic["recall"] = (by_tactic["caught"] / by_tactic["processes"]).round(3)
by_tactic["seen_in_development"] = by_tactic.index.isin(seen_tactics)
by_tactic.sort_values("seen_in_development").round(3)

RES = Path("../../docs/results")
by_tactic.round(3).to_csv(RES / "heldout_by_tactic.csv")

In [5]:
win = pd.DataFrame({"window": (test["t_start"] // 60).astype(int),
                    "attack": y_t.astype(bool), "alert": alert_t}).groupby("window").max()
window_result = {
    "windows_with_activity": len(win),
    "attack_windows": int(win["attack"].sum()),
    "attack_windows_detected": int((win["attack"] & win["alert"]).sum()),
    "false_alert_windows": int((win["alert"] & ~win["attack"]).sum()),
}
window_result["window_recall"] = round(window_result["attack_windows_detected"] / window_result["attack_windows"], 3)
pd.Series(window_result)

windows_with_activity      263.0
attack_windows               9.0
attack_windows_detected      9.0
false_alert_windows          0.0
window_recall                1.0
dtype: float64

In [6]:
novel = (-svm.score_samples(X_t) - cfg["novelty_cutoff"]) >= 0
print("share flagged as novel, by kind:")
print(pd.Series(novel).groupby(kind_t).mean().round(3))

missed = (y_t == 1) & ~alert_t
print("\nattack processes missed by the main detector:", int(missed.sum()))
print("of those, flagged as novel by One-Class SVM:", int((missed & novel).sum()))
print(test.loc[missed, "tactic"].value_counts())

share flagged as novel, by kind:
flood     1.000
normal    0.006
other     0.722
dtype: float64

attack processes missed by the main detector: 13
of those, flagged as novel by One-Class SVM: 4
tactic
defenceEvasion      12
credentialAccess     1
Name: count, dtype: int64


In [7]:
novelty_result = pd.Series({
    "normal_flagged_share": round(novel[kind_t == "normal"].mean(), 3),
    "other_flagged_share": round(novel[kind_t == "other"].mean(), 3),
    "flood_flagged_share": round(novel[kind_t == "flood"].mean(), 3),
    "missed_by_ensemble": int(missed.sum()),
    "missed_but_novel": int((missed & novel).sum()),
})
novelty_result.to_csv(RES / "heldout_novelty.csv")
novelty_result

normal_flagged_share     0.006
other_flagged_share      0.722
flood_flagged_share      1.000
missed_by_ensemble      13.000
missed_but_novel         4.000
dtype: float64

In [9]:
from sklearn.model_selection import StratifiedGroupKFold
from lightgbm import LGBMClassifier
from apt_detection.device_telemetry.features import FEATURE_GROUPS
from apt_detection.device_telemetry.train import cv_scores
from apt_detection.device_telemetry.evaluate import score_report

drop = set(FEATURE_GROUPS["cmdline"] + FEATURE_GROUPS["privilege"] + FEATURE_GROUPS["context"])
BEHAV = [c for c in cfg["features"] if c not in drop]

dev = F2[F2["t_start"] < cutoff]
kind_d = np.where(dev["flood"] == 1, "flood", np.where(dev["label"] == 1, "other", "normal"))
folds = list(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(dev, kind_d, groups=dev["time_block"]))

make = lambda: LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15,
                              min_child_samples=10, random_state=42, verbose=-1)
oof_b = cv_scores(make, dev[BEHAV], dev["label"], kind=kind_d, folds=folds, weighted=False)
thr_b = np.quantile(oof_b[kind_d == "normal"], 1 - 0.002)

lgbm_b = make().fit(dev[BEHAV], dev["label"])
score_b = lgbm_b.predict_proba(test[BEHAV])[:, 1]
behav_result = score_report(y_t, score_b, kind_t, threshold=thr_b)
print(len(BEHAV), "behaviour features, threshold", round(thr_b, 4))
pd.Series(behav_result).round(3)

32 behaviour features, threshold 0.9995


recall_all      0.000
recall_flood    0.000
recall_other    0.000
fpr             0.000
precision       0.000
pr_auc_other    0.498
dtype: float64

In [10]:
print("threshold:", round(thr_b, 4))
print("highest attack score on test:", round(score_b[y_t == 1].max(), 4))
print("highest normal score on test:", round(score_b[y_t == 0].max(), 4))

rows = {}
for fpr_target in [0.002, 0.01, 0.02, 0.05]:
    t = np.quantile(oof_b[kind_d == "normal"], 1 - fpr_target)
    r = score_report(y_t, score_b, kind_t, threshold=t)
    rows[f"dev_fpr_{fpr_target}"] = {"threshold": t, **r}
behav_thresholds = pd.DataFrame(rows).T.round(4)
behav_thresholds

threshold: 0.9995
highest attack score on test: 0.9985
highest normal score on test: 0.9954


,threshold,recall_all,recall_flood,recall_other,fpr,precision,pr_auc_other
dev_fpr_0.002,0.9995,0.0000,0.0,0.0000,0.0000,0.0000,0.4983
dev_fpr_0.01,0.9995,0.0000,0.0,0.0000,0.0000,0.0000,0.4983
dev_fpr_0.02,0.9976,0.0233,0.0,0.0278,0.0000,1.0000,0.4983
dev_fpr_0.05,0.0786,0.7907,1.0,0.7500,0.0145,0.6071,0.4983


In [11]:
dev_normal = dev[kind_d == "normal"].assign(score=oof_b[kind_d == "normal"])
top = dev_normal[dev_normal["score"] >= 0.99]
print(len(top), "of", len(dev_normal), "normal dev processes score >= 0.99")
cols = [c for c in ("name", "cmdline") if c in dev.columns]
top[cols].value_counts().head(10) if cols else top[[c for c in BEHAV if c.startswith("prog_")]].sum().sort_values(ascending=False).head(5)

108 of 3766 normal dev processes score >= 0.99


prog_rarity         340.143523
prog_in_baseline     42.000000
prog_file_op         32.000000
prog_shell            0.000000
prog_interpreter      0.000000
dtype: float64

In [12]:
final_result = score_report(y_t, score_t, kind_t, threshold=cfg["threshold"])

pd.DataFrame({"final_ensemble": final_result, "behaviour_only_lgbm": behav_result}).T.round(4)\
  .to_csv(RES / "heldout_process_level.csv")                      # Cell 2 + Cell 6
by_tactic.round(3).to_csv(RES / "heldout_by_tactic.csv")           # Cell 3
pd.Series(window_result).to_csv(RES / "heldout_window_level.csv")  # Cell 4
pd.Series({
    "normal_flagged_share": round(novel[kind_t == "normal"].mean(), 3),
    "other_flagged_share": round(novel[kind_t == "other"].mean(), 3),
    "flood_flagged_share": round(novel[kind_t == "flood"].mean(), 3),
    "missed_by_ensemble": int(missed.sum()),
    "missed_but_novel": int((missed & novel).sum()),
}).to_csv(RES / "heldout_novelty.csv")                             # Cell 5
behav_thresholds.to_csv(RES / "heldout_behaviour_thresholds.csv")  # Cell 6b

sorted(p.name for p in RES.glob("heldout_*.csv"))

['heldout_behaviour_thresholds.csv',
 'heldout_by_tactic.csv',
 'heldout_novelty.csv',
 'heldout_process_level.csv',
 'heldout_window_level.csv']

In [13]:
print(len(top), "of", len(dev_normal))

108 of 3766
